# Project 2.5 â€” System Identification of a Nonlinear Dynamical System

## 1. Imports and device

In [ ]:

import os, json, time, math, random
from pathlib import Path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch:", torch.__version__)

## 2. Configuration

All hyperparameters in one dict so Phase 2 sweeps are just copies of this with values changed.

In [ ]:
CFG = {
    "fs": 256,                 # sample rate (Hz) after downsampling
    # Dataset + split
    "dataset": "medium",       # "small" | "medium"
    "split":   "loso",         # "within" | "loso"
    "narx_nu": 20,
    "narx_ny": 5,
    # Dense consecutive taps matching Gu 2021.
    "narx_u_taps": tuple(range(0, 20)),     # 0..19
    "narx_y_taps": tuple(range(1, 6)),      # 1..5
    "narx_hidden": (64, 64),  # MLP hidden widths
    # NNARX architecture
    "narx_act":             "gelu",
    "narx_dropout":         0.1,
    # NNARX training
    "narx_val_every":       2,
    "narx_patience_checks": 10,
    "narx_grad_clip":       1.0,
    # k-step curriculum: stage k values + per-stage epoch budget. Advance is
    # fixed-by-epoch (no plateau gating).
    "narx_kstep_schedule":   (1, 5, 10, 20, 40, 80, 160, 320),
    "narx_kstep_min_epochs": (30, 20, 20, 20, 20, 20, 15, 15),
    "narx_lr_schedule":     "cosine",
    "narx_ensemble":        3,
    # Recurrent models
    "rec_hidden": 64,
    "rec_layers": 1,
    "rec_grad_clip":          1.0,
    "rec_warmup":             10,
    "rec_kstep_schedule":    (1, 5, 20, 64, 256, 512, 1024),
    "rec_kstep_min_epochs":  (25, 15, 15, 15, 15, 15, 15),
    "rec_tbptt_len":          0,
    #dropout on hidden state for LSTM/GRU.
    "rec_var_dropout":        0.1,
    # Multi-seed reporting for Phase 2 
    "phase2_seeds":           3,
    # Training
    "epochs_narx": 200,
    "epochs_rec": 200,
    "batch_narx": 512,
    "lr": 1e-3,
    "weight_decay": 0.0,
    # Within-participant split (indices into M=7 realizations per participant)
    "train_real_idx": [0, 1, 2, 3, 4],
    "val_real_idx":   [5],
    "test_real_idx":  [6],
    # LOSO split
    "loso_test_subject":  0,
    "loso_val_real_idx":  [5, 6],
}
CFG


## 3. Load the dataset

In [ ]:
def find_mat_file(filename):
    stem = filename.replace(".mat", "")
    candidates = [Path(stem) / filename, Path(filename)]
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for sub in kaggle_root.iterdir():
            for p in sub.rglob(filename):
                candidates.append(p)
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"{filename} not found in any expected location.")


def load_small(mat_path):
    mat = sio.loadmat(str(mat_path), simplify_cells=True)
    data = mat["data"]
    u = np.asarray(data["u"], dtype=np.float32)   # [S, M, N]
    y = np.asarray(data["y"], dtype=np.float32)
    return u, y


ROLL_SHIFT = 5  # samples; matches MATLAB circshift(u,5,3) input-delay alignment


def load_medium(mat_path, ds=8):
    """Python port of Load_Plot_EEG_2.m: average over periods, downsample,
    per-subject zero-mean/scale, delay input by 5 samples. Output shape [S, M, N]."""
    mat = sio.loadmat(str(mat_path), simplify_cells=True)
    eeg = mat["EEGdata"]  
    u_list, y_list = [], []
    for rec in eeg:
        angle = np.asarray(rec["angle"], dtype=np.float32)  # [M, P, N_raw]
        comp  = np.asarray(rec["comp"],  dtype=np.float32)
        u_list.append(angle[:, :, ::ds].mean(axis=1))       # -> [M, N]
        y_list.append(comp[:,  :, ::ds].mean(axis=1))
    u = np.stack(u_list, axis=0)  # [S, M, N]
    y = np.stack(y_list, axis=0)
    u = u - u.mean(axis=-1, keepdims=True)
    y = y - y.mean(axis=-1, keepdims=True)
    u_scale = u.std(axis=-1, ddof=1).mean(axis=-1, keepdims=True)   # [S, 1]
    y_scale = y.std(axis=-1, ddof=1).mean(axis=-1, keepdims=True)
    u = u / u_scale[:, :, None]
    y = y / y_scale[:, :, None]
    # response is delayed, so also delay the input
    u = np.roll(u, shift=ROLL_SHIFT, axis=-1)
    return u.astype(np.float32), y.astype(np.float32)


if CFG["dataset"] == "small":
    mat_path = find_mat_file("Benchmark_EEG_small.mat")
    print("Loading small:", mat_path)
    u_all, y_all = load_small(mat_path)
elif CFG["dataset"] == "medium":
    mat_path = find_mat_file("Benchmark_EEG_medium.mat")
    print("Loading medium:", mat_path)
    u_all, y_all = load_medium(mat_path)
else:
    raise ValueError(f"unknown dataset {CFG['dataset']!r}")

S, M, N = u_all.shape
print(f"participants S={S}, realizations M={M}, samples N={N}, "
      f"fs={CFG['fs']} Hz -> segment length {N/CFG['fs']:.2f}s")

Quick look at one participant / one realization â€” time domain.

In [ ]:
t = np.arange(N) / CFG["fs"]
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t, u_all[0, 0], lw=1); axes[0].set_ylabel("u (angle, norm.)")
axes[1].plot(t, y_all[0, 0], lw=1); axes[1].set_ylabel("y (EEG, norm.)")
axes[1].set_xlabel("time [s]")
axes[0].set_title("Participant 0 â€” realization 0")
plt.tight_layout(); plt.show()

## 4. Train / val / test split

In [ ]:
def make_split(u_all, y_all, cfg):
    def take(idxs):
        u = u_all[:, idxs, :].reshape(-1, u_all.shape[-1])
        y = y_all[:, idxs, :].reshape(-1, y_all.shape[-1])
        return u, y
    return take(cfg["train_real_idx"]), take(cfg["val_real_idx"]), take(cfg["test_real_idx"])


def make_loso_split(u_all, y_all, cfg):
    S, M, N = u_all.shape
    test_s  = int(cfg["loso_test_subject"])
    val_idx = list(cfg["loso_val_real_idx"])
    tr_idx  = [m for m in range(M) if m not in val_idx]
    # test
    u_te = u_all[test_s].reshape(-1, N)
    y_te = y_all[test_s].reshape(-1, N)
    # train + val
    keep  = [s for s in range(S) if s != test_s]
    u_oth = u_all[keep]   # [S-1, M, N]
    y_oth = y_all[keep]
    u_tr = u_oth[:, tr_idx,  :].reshape(-1, N)
    y_tr = y_oth[:, tr_idx,  :].reshape(-1, N)
    u_va = u_oth[:, val_idx, :].reshape(-1, N)
    y_va = y_oth[:, val_idx, :].reshape(-1, N)
    return (u_tr, y_tr), (u_va, y_va), (u_te, y_te)


if CFG["split"] == "within":
    (u_tr, y_tr), (u_va, y_va), (u_te, y_te) = make_split(u_all, y_all, CFG)
    print(f"[within-participant] train: {u_tr.shape}, val: {u_va.shape}, test: {u_te.shape}")
elif CFG["split"] == "loso":
    (u_tr, y_tr), (u_va, y_va), (u_te, y_te) = make_loso_split(u_all, y_all, CFG)
    print(f"[LOSO  test_subject={CFG['loso_test_subject']}] "
          f"train: {u_tr.shape}, val: {u_va.shape}, test: {u_te.shape}")
else:
    raise ValueError(f"unknown split {CFG['split']!r}")

## 5. Metrics

Following the nonlinear-benchmarks convention:

* **NRMSE** = RMSE(y, Å·) / std(y)
* **VAF** = 100 Â· (1 âˆ’ var(y âˆ’ Å·) / var(y))  (percentage)

In [ ]:
def nrmse(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return float(rmse / (y_true.std() + 1e-12))

def vaf(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return float(100.0 * (1.0 - np.var(y_true - y_pred) / (np.var(y_true) + 1e-12)))

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 6. NNARX

In [ ]:
def _normalize_taps(u_taps, y_taps, nu, ny):
    u_taps = tuple(range(nu)) if u_taps is None else tuple(int(t) for t in u_taps)
    y_taps = tuple(range(1, ny + 1)) if y_taps is None else tuple(int(t) for t in y_taps)
    assert all(t >= 0 for t in u_taps), "u_taps must be non-negative"
    assert all(t >= 1 for t in y_taps), "y_taps must be >= 1"
    return u_taps, y_taps


def build_narx_windows(u_seqs, y_seqs, u_taps, y_taps):
    u_taps_arr = np.asarray(u_taps, dtype=np.int64)
    y_taps_arr = np.asarray(y_taps, dtype=np.int64)
    Lu = int(u_taps_arr.max()) + 1
    Ly = int(y_taps_arr.max()) if len(y_taps_arr) else 0
    L = max(Lu, Ly)
    xs, ts = [], []
    for u, y in zip(u_seqs, y_seqs):
        u = np.asarray(u, dtype=np.float32)
        Nseq = len(u)
        ks = np.arange(max(L, ROLL_SHIFT), Nseq)  
        u_idx = ks[:, None] - u_taps_arr[None, :]
        u_lags = u[u_idx]
        if Ly > 0:
            y_idx = ks[:, None] - y_taps_arr[None, :]
            y_lags = y[y_idx]
            xs.append(np.concatenate([u_lags, y_lags], axis=-1))
        else:
            xs.append(u_lags)
        ts.append(y[ks])
    X = np.concatenate(xs).astype(np.float32)
    t = np.concatenate(ts).astype(np.float32)[:, None]
    return X, t


def build_narx_rollout(u_seqs, y_seqs, u_taps, y_taps, kstep):
    u_taps_arr = np.asarray(u_taps, dtype=np.int64)
    y_taps_arr = np.asarray(y_taps, dtype=np.int64)
    Lu = int(u_taps_arr.max()) + 1
    Ly = int(y_taps_arr.max()) if len(y_taps_arr) else 0
    L = max(Lu, Ly)
    Us, Yw, Yt = [], [], []
    for u, y in zip(u_seqs, y_seqs):
        u = np.asarray(u, dtype=np.float32)
        Nseq = len(u)
        ks = np.arange(max(L, ROLL_SHIFT), Nseq - kstep + 1)  # skip np.roll wrap zone
        if len(ks) == 0:
            continue
        win_len = Lu + kstep - 1
        win_idx = (ks - Lu + 1)[:, None] + np.arange(win_len)[None, :]
        Us.append(u[win_idx])
        if Ly > 0:
            yw_idx = (ks - Ly)[:, None] + np.arange(Ly)[None, :]
            Yw.append(y[yw_idx])
        else:
            Yw.append(np.zeros((len(ks), 0), dtype=np.float32))
        yt_idx = ks[:, None] + np.arange(kstep)[None, :]
        Yt.append(y[yt_idx])
    return (np.concatenate(Us).astype(np.float32),
            np.concatenate(Yw).astype(np.float32),
            np.concatenate(Yt).astype(np.float32))


_ACTS = {"tanh": nn.Tanh, "relu": nn.ReLU, "gelu": nn.GELU}


class NNARX(nn.Module):
    def __init__(self, u_taps, y_taps, hidden=(64, 64), act="tanh", dropout=0.0):
        super().__init__()
        self.u_taps = tuple(u_taps)
        self.y_taps = tuple(y_taps)
        self.nu = len(self.u_taps)
        self.ny = len(self.y_taps)
        self.Lu = (max(self.u_taps) + 1) if self.u_taps else 0
        self.Ly = max(self.y_taps) if self.y_taps else 0
        self.L  = max(self.Lu, self.Ly)
        in_dim = self.nu + self.ny
        act_cls = _ACTS[act.lower()]
        layers = []
        dims = [in_dim] + list(hidden)
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), act_cls()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(dims[-1], 1)]
        self.net = nn.Sequential(*layers)
        self.in_dim = in_dim
        self.register_buffer("_u_taps_t",
            torch.as_tensor(self.u_taps, dtype=torch.long), persistent=False)
        self.register_buffer("_y_taps_t",
            torch.as_tensor(self.y_taps, dtype=torch.long), persistent=False)

    def forward(self, x):
        return self.net(x)

    @torch.no_grad()
    def simulate(self, u_seq, y_warmup):
        import scipy.special as _ss
        _was_training = self.training
        self.eval()  # disable Dropout 
        try:
            return self._simulate_impl(u_seq, y_warmup, _ss)
        finally:
            self.train(_was_training)

    def _simulate_impl(self, u_seq, y_warmup, _ss):
        u = np.asarray(u_seq, dtype=np.float32)
        Nseq = len(u_seq); L = self.L
        y_hat = np.zeros(Nseq, dtype=np.float32)
        y_hat[:L] = np.asarray(y_warmup, dtype=np.float32)[:L]
        u_taps_np = np.array(self.u_taps, dtype=np.int64)
        y_taps_np = np.array(self.y_taps, dtype=np.int64)
        np_layers = []
        for m in self.net:
            if isinstance(m, nn.Linear):
                np_layers.append(('lin',
                    m.weight.detach().cpu().float().numpy(),
                    m.bias.detach().cpu().float().numpy()))
            elif isinstance(m, nn.Tanh):
                np_layers.append(('tanh',))
            elif isinstance(m, nn.ReLU):
                np_layers.append(('relu',))
            elif isinstance(m, nn.GELU):
                np_layers.append(('gelu',))
        _sqrt2 = 1.4142135623730951

        def _fwd(x1d):
            h = x1d
            for lyr in np_layers:
                if lyr[0] == 'lin':
                    h = h @ lyr[1].T + lyr[2]
                elif lyr[0] == 'tanh':
                    h = np.tanh(h)
                elif lyr[0] == 'relu':
                    h = np.maximum(0.0, h)
                elif lyr[0] == 'gelu':
                    h = h * 0.5 * (1.0 + _ss.erf(h / _sqrt2))
            return float(h[0])

        k0 = max(L, ROLL_SHIFT)
        if k0 > L:
            y_hat[L:k0] = np.asarray(y_warmup, dtype=np.float32)[L:k0]
        # one-step parity check vs torch forward (regression guard).
        if k0 < Nseq:
            u_lags0 = u[k0 - u_taps_np]
            y_lags0 = y_hat[k0 - y_taps_np] if self.Ly > 0 else None
            x0 = np.concatenate([u_lags0, y_lags0]) if self.Ly > 0 else u_lags0
            y_np0 = _fwd(x0.astype(np.float32))
            dev = next(self.parameters()).device
            x0_t = torch.from_numpy(x0[None, :].astype(np.float32)).to(dev)
            y_t0 = float(self.net(x0_t).squeeze().item())
            assert abs(y_np0 - y_t0) < 1e-4, (
                f'NNARX.simulate numpy/torch parity broken: |{y_np0} - {y_t0}| >= 1e-4')
        for k in range(k0, Nseq):
            u_lags = u[k - u_taps_np]
            if self.Ly > 0:
                y_lags = y_hat[k - y_taps_np]
                x = np.concatenate([u_lags, y_lags])
            else:
                x = u_lags
            y_hat[k] = _fwd(x)
        return y_hat

    @torch.no_grad()
    def simulate_osa(self, u_seq, y_seq):
        """One-step-ahead: regressor uses true past y at every step."""
        dev = next(self.parameters()).device
        u_gpu = torch.as_tensor(np.asarray(u_seq, dtype=np.float32)).to(dev)
        y_gpu = torch.as_tensor(np.asarray(y_seq, dtype=np.float32)).to(dev)
        Nseq = len(u_seq); L = self.L
        y_hat = torch.zeros(Nseq, device=dev)
        y_hat[:L] = y_gpu[:L]
        self.eval()
        k0 = max(L, ROLL_SHIFT)
        if k0 > L:
            y_hat[L:k0] = y_gpu[L:k0]
        for k in range(k0, Nseq):
            u_lags = u_gpu[k - self._u_taps_t].unsqueeze(0)
            if self.Ly > 0:
                y_lags = y_gpu[k - self._y_taps_t].unsqueeze(0)
                x = torch.cat([u_lags, y_lags], dim=-1)
            else:
                x = u_lags
            y_hat[k] = self.net(x).squeeze()
        return y_hat.cpu().numpy()

    def rollout(self, U_win, Y_warm, kstep):
        """U_win [B, Lu+kstep-1], Y_warm [B, Ly] -> preds [B, kstep]."""
        B = U_win.size(0)
        u_taps = self._u_taps_t
        y_taps = self._y_taps_t
        y_buf = Y_warm
        preds = []
        for j in range(kstep):
            idx_u = (self.Lu - 1 + j) - u_taps
            u_slice = U_win.index_select(1, idx_u)
            if self.Ly > 0:
                idx_y = self.Ly - y_taps
                y_slice = y_buf.index_select(1, idx_y)
                x = torch.cat([u_slice, y_slice], dim=-1)
            else:
                x = u_slice
            y_pred = self.net(x).squeeze(-1)
            preds.append(y_pred)
            if self.Ly > 0:
                y_buf = torch.cat([y_buf[:, 1:], y_pred.unsqueeze(-1)], dim=-1)
        return torch.stack(preds, dim=-1)


## 7. Simple RNN / LSTM / GRU

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, hidden=64, layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size=2, hidden_size=hidden, num_layers=layers,
                          nonlinearity="tanh", batch_first=True)
        self.head = nn.Linear(hidden, 1)
        self.hidden = hidden
        self.layers = layers

    def forward(self, u):
        h, _ = self.rnn(u)
        return self.head(h)

In [ ]:
class _HandLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, var_dropout=0.0):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.var_dropout = float(var_dropout)
        self._fixed_masks = None  # locked variational dropout masks
        self.W_ih = nn.ModuleList()
        self.W_hh = nn.ModuleList()
        for l in range(num_layers):
            in_sz = input_size if l == 0 else hidden_size
            self.W_ih.append(nn.Linear(in_sz, 4 * hidden_size, bias=True))
            self.W_hh.append(nn.Linear(hidden_size, 4 * hidden_size, bias=True))

    def forward(self, x, hc=None):
        B, T, _ = x.shape
        H, L = self.hidden_size, self.num_layers
        if hc is None:
            h = x.new_zeros(L, B, H)
            c = x.new_zeros(L, B, H)
        else:
            h, c = hc
        h_list = [h[l] for l in range(L)]
        c_list = [c[l] for l in range(L)]
        if self._fixed_masks is not None:
            masks = self._fixed_masks
        elif self.training and self.var_dropout > 0.0:
            keep = 1.0 - self.var_dropout
            masks = [(torch.rand(B, H, device=x.device) < keep).float() / keep
                     for _ in range(L)]
        else:
            masks = None
        outs = []
        for t in range(T):
            layer_in = x[:, t, :]
            for l in range(L):
                h_in = h_list[l] if masks is None else h_list[l] * masks[l]
                gates = self.W_ih[l](layer_in) + self.W_hh[l](h_in)
                i, f, g, o = gates.chunk(4, dim=-1)
                i = torch.sigmoid(i); f = torch.sigmoid(f)
                g = torch.tanh(g);    o = torch.sigmoid(o)
                c_new = f * c_list[l] + i * g
                h_new = o * torch.tanh(c_new)
                h_list[l] = h_new
                c_list[l] = c_new
                layer_in = h_new
            outs.append(layer_in)
        out = torch.stack(outs, dim=1)
        return out, (torch.stack(h_list, dim=0), torch.stack(c_list, dim=0))


class LSTMModel(nn.Module):
    def __init__(self, hidden=64, layers=1, var_dropout=0.0):
        super().__init__()
        self.rnn = _HandLSTM(input_size=2, hidden_size=hidden, num_layers=layers,
                             var_dropout=var_dropout)
        self.head = nn.Linear(hidden, 1)
        self.hidden = hidden
        self.layers = layers

    def forward(self, u):
        h, _ = self.rnn(u)
        return self.head(h)


In [ ]:
class _HandGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, var_dropout=0.0):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.var_dropout = float(var_dropout)
        self._fixed_masks = None  # locked variational dropout masks (set by train loop)
        self.W_ih = nn.ModuleList()
        self.W_hh = nn.ModuleList()
        for l in range(num_layers):
            in_sz = input_size if l == 0 else hidden_size
            self.W_ih.append(nn.Linear(in_sz, 3 * hidden_size, bias=True))
            self.W_hh.append(nn.Linear(hidden_size, 3 * hidden_size, bias=True))

    def forward(self, x, h=None):
        B, T, _ = x.shape
        H, L = self.hidden_size, self.num_layers
        if h is None:
            h = x.new_zeros(L, B, H)
        h_list = [h[l] for l in range(L)]
        if self._fixed_masks is not None:
            masks = self._fixed_masks
        elif self.training and self.var_dropout > 0.0:
            keep = 1.0 - self.var_dropout
            masks = [(torch.rand(B, H, device=x.device) < keep).float() / keep
                     for _ in range(L)]
        else:
            masks = None
        outs = []
        for t in range(T):
            layer_in = x[:, t, :]
            for l in range(L):
                h_in = h_list[l] if masks is None else h_list[l] * masks[l]
                xg = self.W_ih[l](layer_in)
                hg = self.W_hh[l](h_in)
                x_r, x_z, x_n = xg.chunk(3, dim=-1)
                h_r, h_z, h_n = hg.chunk(3, dim=-1)
                r = torch.sigmoid(x_r + h_r)
                z = torch.sigmoid(x_z + h_z)
                n = torch.tanh(x_n + r * h_n)
                h_new = (1.0 - z) * n + z * h_in  # BUG-5: mask-consistent residual
                h_list[l] = h_new
                layer_in = h_new
            outs.append(layer_in)
        return torch.stack(outs, dim=1), torch.stack(h_list, dim=0)


class GRUModel(nn.Module):
    def __init__(self, hidden=64, layers=1, var_dropout=0.0):
        super().__init__()
        self.rnn = _HandGRU(input_size=2, hidden_size=hidden, num_layers=layers,
                            var_dropout=var_dropout)
        self.head = nn.Linear(hidden, 1)
        self.hidden = hidden
        self.layers = layers

    def forward(self, u):
        h, _ = self.rnn(u)
        return self.head(h)


## 8. Training loops

In [ ]:
def _resolve_kstep_schedule(kstep_schedule, kstep_min_epochs):
    if not kstep_schedule:
        return [(0, 1)]
    norm = []
    cum = 0
    for i, item in enumerate(kstep_schedule):
        if isinstance(item, (tuple, list)):
            thr, k = item
            norm.append((int(thr), int(k)))
        else:
            if i == 0:
                thr = 0
            else:
                if isinstance(kstep_min_epochs, (tuple, list)):
                    dur = int(kstep_min_epochs[min(i - 1, len(kstep_min_epochs) - 1)])
                else:
                    dur = int(kstep_min_epochs) if kstep_min_epochs else 0
                cum += dur
                thr = cum
            norm.append((thr, int(item)))
    return sorted(norm)


def train_narx(model, u_tr, y_tr, u_va, y_va, epochs, batch, lr, weight_decay=0.0,
               val_every=5, patience_checks=5,
               kstep_schedule=((0, 1),), kstep_min_epochs=None,
               lr_schedule=None, grad_clip=None):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
                 if lr_schedule == "cosine" else None)
    loss_fn = nn.MSELoss()

    _pin  = torch.cuda.is_available()
    _on_kaggle = os.path.exists("/kaggle")
    _nwrk = 0 if _on_kaggle else (2 if _pin else 0)

    X_tr1, t_tr1 = build_narx_windows(u_tr, y_tr, model.u_taps, model.y_taps)
    dl1 = DataLoader(TensorDataset(torch.from_numpy(X_tr1), torch.from_numpy(t_tr1)),
                     batch_size=batch, shuffle=True, drop_last=False,
                     pin_memory=_pin, num_workers=_nwrk, persistent_workers=(_nwrk > 0))

    rollout_cache = {}
    def get_rollout_loader(k):
        if k in rollout_cache: return rollout_cache[k]
        U, Yw, Yt = build_narx_rollout(u_tr, y_tr, model.u_taps, model.y_taps, k)
        dl = DataLoader(TensorDataset(torch.from_numpy(U), torch.from_numpy(Yw),
                                      torch.from_numpy(Yt)),
                        batch_size=batch, shuffle=True, drop_last=False,
                        pin_memory=_pin, num_workers=_nwrk, persistent_workers=(_nwrk > 0))
        rollout_cache[k] = dl
        return dl

    sched = _resolve_kstep_schedule(kstep_schedule, kstep_min_epochs)
    def current_kstep(ep):
        cur = sched[0][1]
        for thr, k in sched:
            if ep >= thr: cur = k
        return cur

    history = {"train": [], "val_sim_nrmse": [], "kstep": []}
    best_nrmse, best_state, bad_checks = float("inf"), None, 0
    prev_kstep = None
    t0 = time.time()

    for ep in range(epochs):
        kstep = current_kstep(ep)
        if prev_kstep is not None and kstep != prev_kstep:
            bad_checks = 0
            print(f"  [curriculum] k {prev_kstep} -> {kstep}")
        prev_kstep = kstep
        history["kstep"].append(kstep)
        model.train(); losses = []
        if kstep <= 1:
            for xb, tb in dl1:
                xb, tb = xb.to(DEVICE), tb.to(DEVICE)
                opt.zero_grad()
                loss = loss_fn(model(xb), tb); loss.backward()
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()
                losses.append(loss.item())
        else:
            dl_k = get_rollout_loader(kstep)
            for U_b, Yw_b, Yt_b in dl_k:
                U_b  = U_b.to(DEVICE)
                Yw_b = Yw_b.to(DEVICE)
                Yt_b = Yt_b.to(DEVICE)
                opt.zero_grad()
                preds = model.rollout(U_b, Yw_b, kstep)
                loss = loss_fn(preds, Yt_b)
                loss.backward()
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()
                losses.append(loss.item())

        if scheduler is not None: scheduler.step()
        history["train"].append(float(np.mean(losses)))

        if (ep + 1) % val_every == 0 or ep == epochs - 1:
            y_hat, y_true = eval_narx(model, u_va, y_va)
            v_nrmse = nrmse(y_true, y_hat)
            history["val_sim_nrmse"].append((ep + 1, v_nrmse))
            improved = v_nrmse < best_nrmse - 1e-4
            if improved:
                best_nrmse = v_nrmse
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                bad_checks = 0; tag = "(best)"
   
            else:
                bad_checks += 1; tag = f"(no improve {bad_checks}/{patience_checks})"
            lr_now = opt.param_groups[0]["lr"]
            print(f"ep {ep+1:3d}/{epochs}  k={kstep}  lr={lr_now:.2e}  "
                  f"train {history['train'][-1]:.4e}  val sim NRMSE {v_nrmse:.4f}  {tag}")
            if bad_checks >= patience_checks:
                print(f"early stop at epoch {ep+1} (best val sim NRMSE = {best_nrmse:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, time.time() - t0


def train_seq(model, u_tr, y_tr, u_va, y_va, epochs, lr, weight_decay=0.0,
              val_every=5, patience=20, grad_clip=1.0, warmup=10,
              kstep_schedule=((0, 1),), kstep_min_epochs=None, tbptt_len=None):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()
    U_tr = torch.from_numpy(u_tr).float().unsqueeze(-1).to(DEVICE)
    Y_tr = torch.from_numpy(y_tr).float().unsqueeze(-1).to(DEVICE)
    B, T, _ = U_tr.shape
    Y_prev_tr = torch.cat([torch.zeros_like(Y_tr[:, :1]), Y_tr[:, :-1]], dim=1)

    sched = _resolve_kstep_schedule(kstep_schedule, kstep_min_epochs)
    def current_kstep(ep):
        cur = sched[0][1]
        for thr, k in sched:
            if ep >= thr: cur = k
        return cur

    history = {"train": [], "val_nrmse": [], "kstep": []}
    best_nrmse, best_state, bad = float("inf"), None, 0
    prev_kstep = None
    t0 = time.time()

    for ep in range(epochs):
        kstep = current_kstep(ep)
        if prev_kstep is not None and kstep != prev_kstep:
            bad = 0
            print(f"  [curriculum] k {prev_kstep} -> {kstep}")
        prev_kstep = kstep
        history["kstep"].append(kstep)
        chunk = int(tbptt_len) if (tbptt_len is not None and int(tbptt_len) > 0) else kstep

        # Lock variational-dropout masks for this whole epoch so that
        # per-timestep model.rnn calls in the k>1 branch reuse the same mask.
        if hasattr(model.rnn, '_fixed_masks') and getattr(model.rnn, 'var_dropout', 0.0) > 0.0:
            _kh = model.rnn.hidden_size; _kl = model.rnn.num_layers
            _keep = 1.0 - model.rnn.var_dropout
            model.rnn._fixed_masks = [
                (torch.rand(B, _kh, device=DEVICE) < _keep).float() / _keep
                for _ in range(_kl)]
        model.train(); opt.zero_grad()
        if kstep <= 1:
            X = torch.cat([U_tr, Y_prev_tr], dim=-1)
            loss = loss_fn(model(X), Y_tr)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            losses_ep = loss.item()
        else:
            h = None
            preds = []
            chunk_losses = []
            last_pred = torch.zeros(B, 1, device=DEVICE)
            chunk_start = 0
            for t in range(T):
                if t == 0:
                    y_prev = torch.zeros(B, 1, device=DEVICE)
                elif t % kstep == 0:
                    y_prev = Y_tr[:, t - 1, :]
                else:
                    y_prev = last_pred
                u_t = U_tr[:, t, :]
                x_t = torch.cat([u_t, y_prev], dim=-1).unsqueeze(1)
                out, h = model.rnn(x_t, h)
                yp = model.head(out.squeeze(1))
                preds.append(yp)
                last_pred = yp
                if ((t + 1) - chunk_start) >= chunk or t == T - 1:
                    p_chunk = torch.stack(preds, dim=1)
                    y_chunk = Y_tr[:, chunk_start:t + 1, :]
                    loss_chunk = loss_fn(p_chunk, y_chunk)
                    loss_chunk.backward()
                    if grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    opt.step(); opt.zero_grad()
                    chunk_losses.append(loss_chunk.item())
                    chunk_start = t + 1
                    preds = []
                    if isinstance(h, tuple):
                        h = (h[0].detach(), h[1].detach())
                    elif h is not None:
                        h = h.detach()
                    last_pred = last_pred.detach()
            losses_ep = float(np.mean(chunk_losses)) if chunk_losses else float("nan")
        if hasattr(model.rnn, '_fixed_masks'):
            model.rnn._fixed_masks = None
        scheduler.step()
        history["train"].append(losses_ep)

        if (ep + 1) % val_every == 0 or ep == epochs - 1:
            y_hat, y_true = simulate_seq(model, u_va, y_va, warmup=warmup)
            v_nrmse = nrmse(y_true, y_hat)
            history["val_nrmse"].append((ep + 1, v_nrmse))
            improved = v_nrmse < best_nrmse - 1e-4
            if improved:
                best_nrmse = v_nrmse
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                bad = 0; tag = "(best)"
            else:
                bad += 1; tag = f"(no improve {bad}/{patience})"
            lr_now = opt.param_groups[0]["lr"]
            print(f"ep {ep+1:3d}/{epochs}  k={kstep} chunk={chunk}  "
                  f"lr={lr_now:.2e}  train {history['train'][-1]:.4e}"
                  f"  val sim NRMSE {v_nrmse:.4f}  {tag}")
            if bad >= patience:
                print(f"early stop at epoch {ep+1} (best val sim NRMSE = {best_nrmse:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, time.time() - t0


## 9. Simulation-mode evaluation

In [ ]:
def eval_narx(model, u_seqs, y_seqs):
    preds, truths = [], []
    for u, y in zip(u_seqs, y_seqs):
        L = model.L                      # warmup = max lag span (Lu, Ly), not tap count
        preds.append(model.simulate(u, y[:L]))
        truths.append(np.asarray(y, dtype=np.float32))
    return np.concatenate(preds), np.concatenate(truths)

def eval_seq(model, u_seqs, y_seqs):
    model.eval()
    U = torch.from_numpy(u_seqs).float().unsqueeze(-1).to(DEVICE)
    Y = torch.from_numpy(y_seqs).float().unsqueeze(-1).to(DEVICE)
    Y_prev = torch.cat([torch.zeros_like(Y[:, :1]), Y[:, :-1]], dim=1)
    X = torch.cat([U, Y_prev], dim=-1)
    with torch.no_grad():
        pred = model(X).squeeze(-1).cpu().numpy()
    return pred.reshape(-1), np.asarray(y_seqs, dtype=np.float32).reshape(-1)


def simulate_seq(model, u_seqs, y_seqs, warmup=10):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for u_np, y_np in zip(u_seqs, y_seqs):
            T = len(u_np)
            u = torch.from_numpy(np.asarray(u_np, dtype=np.float32)).to(DEVICE)
            y = torch.from_numpy(np.asarray(y_np, dtype=np.float32)).to(DEVICE)
            y_hat = torch.zeros(T, device=DEVICE)
            h = None
            for k in range(T):
                if k == 0:
                    y_prev = torch.zeros(1, device=DEVICE)
                elif k <= warmup:
                    y_prev = y[k - 1:k]     # teacher-forced warm-up
                else:
                    y_prev = y_hat[k - 1:k]  # free-run
                x_k = torch.stack([u[k:k+1], y_prev], dim=-1).view(1, 1, 2)
                out, h = model.rnn(x_k, h)
                y_hat[k] = model.head(out.view(1, -1)).squeeze()
            all_preds.append(y_hat.cpu().numpy())
            all_trues.append(np.asarray(y_np, dtype=np.float32))
    return np.concatenate(all_preds), np.concatenate(all_trues)


def _detach_state(h):
    if h is None:
        return None
    if isinstance(h, tuple):
        return tuple(x.detach().clone() for x in h)
    return h.detach().clone()


@torch.no_grad()
def eval_3step_narx(model, u_seqs, y_seqs, k=3, batch=4096):
    model.eval()
    U, Yw, Yt = build_narx_rollout(u_seqs, y_seqs, model.u_taps, model.y_taps, k)
    preds = []
    for i in range(0, len(U), batch):
        Ub = torch.from_numpy(U[i:i+batch]).to(DEVICE)
        Wb = torch.from_numpy(Yw[i:i+batch]).to(DEVICE)
        out = model.rollout(Ub, Wb, k)[:, -1]
        preds.append(out.detach().cpu().numpy())
    y_hat  = np.concatenate(preds).astype(np.float32)
    y_true = Yt[:, -1].astype(np.float32)
    return y_hat, y_true


@torch.no_grad()
def kstep_seq(model, u_seqs, y_seqs, k=3):
    model.eval()
    all_preds, all_trues = [], []
    for u_np, y_np in zip(u_seqs, y_seqs):
        T = len(u_np)
        u = torch.from_numpy(np.asarray(u_np, dtype=np.float32)).to(DEVICE)
        y = torch.from_numpy(np.asarray(y_np, dtype=np.float32)).to(DEVICE)
        h = None
        h_trace = [None] * T
        # Teacher-forced trunk
        for t in range(T):
            y_prev = (torch.zeros(1, device=DEVICE) if t == 0 else y[t-1:t])
            x_t = torch.stack([u[t:t+1], y_prev], dim=-1).view(1, 1, 2)
            _, h = model.rnn(x_t, h)
            h_trace[t] = _detach_state(h)
        # k-step rollout per t
        y_hat_k = torch.zeros(T, device=DEVICE)
        for t in range(k, T):
            h_branch = _detach_state(h_trace[t - k])
            y_prev = y[t - k:t - k + 1]
            for j in range(k):
                idx = t - k + 1 + j
                x_j = torch.stack([u[idx:idx + 1], y_prev], dim=-1).view(1, 1, 2)
                out, h_branch = model.rnn(x_j, h_branch)
                y_prev = model.head(out.view(1, -1)).view(1)
            y_hat_k[t] = y_prev
        all_preds.append(y_hat_k[k:].cpu().numpy())
        all_trues.append(np.asarray(y_np[k:], dtype=np.float32))
    return np.concatenate(all_preds), np.concatenate(all_trues)


## 10. Parameters and FLOPs per output sample

In [ ]:
def flops_rnn(model):
    H, L = model.hidden, model.layers
    gates = 1
    f_layer1 = 2 * gates * (2 * H + H * H) + gates * H
    f_other  = 2 * gates * (H * H + H * H) + gates * H
    total = f_layer1 + (L - 1) * f_other
    total += 2 * H + 1          # linear head: H multiplies + bias
    return total

In [ ]:
def flops_lstm(model):
    H, L = model.hidden, model.layers
    gates = 4
    f_layer1 = 2 * gates * (2 * H + H * H) + gates * H
    f_other  = 2 * gates * (H * H + H * H) + gates * H
    total = f_layer1 + (L - 1) * f_other
    total += 5 * H              # nonlinearities + hadamards
    total += 2 * H + 1
    return total


In [ ]:
def flops_gru(model):
    H, L = model.hidden, model.layers
    gates = 3
    f_layer1 = 2 * gates * (2 * H + H * H) + gates * H
    f_other  = 2 * gates * (H * H + H * H) + gates * H
    total = f_layer1 + (L - 1) * f_other
    total += 3 * H
    total += 2 * H + 1
    return total


In [ ]:
def flops_per_sample(model):
    if isinstance(model, NNARX):
        in_dim = model.nu + model.ny
        f, prev = 0, in_dim
        for layer in model.net:
            if isinstance(layer, nn.Linear):
                f += 2 * prev * layer.out_features + layer.out_features
                prev = layer.out_features
        return f
    if isinstance(model, RNNModel):
        return flops_rnn(model)
    if isinstance(model, LSTMModel):
        return flops_lstm(model)
    if isinstance(model, GRUModel):
        return flops_gru(model)
    return None


## 11. Phase: run one model end-to-end

In [ ]:
def plot_sim(y_true, y_pred, title, n=None, y_osa=None):
    if n is None:
        n = min(len(y_true), 4 * int(CFG["fs"]))
    t = np.arange(n) / CFG["fs"]
    plt.figure(figsize=(10, 3))
    plt.plot(t, y_true[:n], label="true", lw=1)
    plt.plot(t, y_pred[:n], label="sim",  lw=1, alpha=0.8)
    if y_osa is not None:
        plt.plot(t, y_osa[:n], label="OSA", lw=1, alpha=0.8, linestyle="--")
    plt.xlabel("time [s]"); plt.ylabel("y"); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.show()

def plot_three_sims(model_name, y_true, y_sim, y_osa,
                    y_true_3s, y_3s,
                    vaf_sim, vaf_osa, vaf_3step, n=None):
    """Three separate plots (OSA / 3-step / free-run sim), each with VAF in title."""
    pairs = [
        ("OSA",    y_true,    y_osa, vaf_osa),
        ("3-step", y_true_3s, y_3s,  vaf_3step),
        ("sim",    y_true,    y_sim, vaf_sim),
    ]
    for label, yt, yp, v in pairs:
        m = min(len(yt), 4 * int(CFG["fs"])) if n is None else n
        t = np.arange(m) / CFG["fs"]
        plt.figure(figsize=(10, 3))
        plt.plot(t, yt[:m], label="true", lw=1)
        plt.plot(t, yp[:m], label=label,  lw=1, alpha=0.8)
        plt.xlabel("time [s]"); plt.ylabel("y")
        plt.title(f"{model_name} -- {label} on test (VAF={v:.2f}%)")
        plt.legend(); plt.tight_layout(); plt.show()


def plot_error_spectrum(y_true, y_hat, fs, title="Error spectrum (FFT of residual)"):
    err = np.asarray(y_true, dtype=np.float64) - np.asarray(y_hat, dtype=np.float64)
    n   = len(err)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    spec  = np.abs(np.fft.rfft(err)) / n
    plt.figure(figsize=(8, 3))
    plt.semilogy(freqs, spec)
    plt.xlabel("frequency [Hz]"); plt.ylabel("|FFT(y - yhat)| / N")
    plt.title(title); plt.tight_layout(); plt.show()


RESULTS = {}


def _make_nnarx(cfg):
    u_taps, y_taps = _normalize_taps(cfg.get("narx_u_taps"),
                                     cfg.get("narx_y_taps"),
                                     cfg["narx_nu"], cfg["narx_ny"])
    return NNARX(u_taps, y_taps,
                 hidden=tuple(cfg["narx_hidden"]),
                 act=cfg.get("narx_act", "tanh"),
                 dropout=cfg.get("narx_dropout", 0.0))


def _train_one_nnarx(cfg, seed_offset):
    seed_base = int(cfg.get("narx_seed_base", SEED))
    s = seed_base + int(seed_offset)
    random.seed(s); torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    model = _make_nnarx(cfg)
    history, elapsed = train_narx(
        model, u_tr, y_tr, u_va, y_va,
        epochs=cfg["epochs_narx"], batch=cfg["batch_narx"],
        lr=cfg["lr"],
        weight_decay=cfg.get("weight_decay", 0.0),
        val_every=cfg.get("narx_val_every", 5),
        patience_checks=cfg.get("narx_patience_checks", 5),
        kstep_schedule=cfg.get("narx_kstep_schedule", ((0, 1),)),
        kstep_min_epochs=cfg.get("narx_kstep_min_epochs", None),
        lr_schedule=cfg.get("narx_lr_schedule", None),
        grad_clip=cfg.get("narx_grad_clip", None),
    )
    return model, history, elapsed


def run_nnarx(cfg, tag="nnarx"):
    n_ens = int(cfg.get("narx_ensemble", 1))
    sim_per_model = []
    trained_models = []
    total_t = 0.0
    last_model, last_history = None, None
    for i in range(n_ens):
        if n_ens > 1:
            print(f"\n[ensemble member {i+1}/{n_ens}]")
        model, history, elapsed = _train_one_nnarx(cfg, seed_offset=i)
        if i == 0:
            print(f"NNARX params: {count_params(model):,}  "
                  f"act={cfg.get('narx_act','tanh')}  ensemble={n_ens}")
            print(f"  u_taps={model.u_taps}  y_taps={model.y_taps}  "
                  f"L_warm={model.L} (Lu={model.Lu}, Ly={model.Ly})")
        seq_preds = []
        for u_seq, y_seq in zip(u_te, y_te):
            L = model.L
            seq_preds.append(model.simulate(u_seq, y_seq[:L]))
        sim_per_model.append(np.concatenate(seq_preds))
        trained_models.append(model)
        total_t += elapsed
        last_model, last_history = model, history

    y_hat = np.mean(np.stack(sim_per_model, axis=0), axis=0)
    y_true = np.concatenate([np.asarray(y, dtype=np.float32) for y in y_te])

    osa_per_model = []
    for mdl in trained_models:
        seq_osa = []
        for u_seq, y_seq in zip(u_te, y_te):
            seq_osa.append(mdl.simulate_osa(u_seq, y_seq))
        osa_per_model.append(np.concatenate(seq_osa))
    y_hat_osa = np.mean(np.stack(osa_per_model, axis=0), axis=0)

    three_step_per_model, y_true_3step = [], None
    for mdl in trained_models:
        yh, y_true_3step = eval_3step_narx(mdl, u_te, y_te, k=3)
        three_step_per_model.append(yh)
    y_hat_3step = np.mean(np.stack(three_step_per_model, axis=0), axis=0)

    res = dict(model="NNARX", params=count_params(last_model),
               flops_per_sample=flops_per_sample(last_model) * n_ens,
               train_time_s=total_t, ensemble=n_ens,
               nrmse_sim=nrmse(y_true, y_hat), vaf_sim=vaf(y_true, y_hat),
               nrmse_osa=nrmse(y_true, y_hat_osa), vaf_osa=vaf(y_true, y_hat_osa),
               nrmse_3step=nrmse(y_true_3step, y_hat_3step),
               vaf_3step=vaf(y_true_3step, y_hat_3step),
               history=last_history)
    print(f"test sim NRMSE={res['nrmse_sim']:.4f}, VAF={res['vaf_sim']:.2f}%, "
          f"train_time={total_t:.1f}s, FLOPs/sample={res['flops_per_sample']:,}")
    print(f"  OSA    NRMSE={res['nrmse_osa']:.4f}, VAF={res['vaf_osa']:.2f}%")
    print(f"  3-step NRMSE={res['nrmse_3step']:.4f}, VAF={res['vaf_3step']:.2f}%")
    plot_three_sims("NNARX", y_true, y_hat, y_hat_osa,
                    y_true_3step, y_hat_3step,
                    res["vaf_sim"], res["vaf_osa"], res["vaf_3step"])
    plot_error_spectrum(y_true, y_hat, cfg["fs"], title=f"{tag} -- error spectrum")
    return last_model, res


In [ ]:
def run_rnn(cfg):
    """Phase 1 runner for RNN."""
    model = RNNModel(hidden=cfg["rec_hidden"], layers=cfg["rec_layers"])
    print(f"RNN params: {count_params(model):,}")
    history, elapsed = train_seq(model, u_tr, y_tr, u_va, y_va,
                                 epochs=cfg["epochs_rec"],
                                 lr=cfg["lr"], weight_decay=cfg["weight_decay"],
                                 grad_clip=cfg.get("rec_grad_clip", 1.0),
                                 warmup=cfg.get("rec_warmup", 10),
                                 kstep_schedule=cfg.get("rec_kstep_schedule", ((0, 1),)),
                                 kstep_min_epochs=cfg.get("rec_kstep_min_epochs", None),
                                 tbptt_len=cfg.get("rec_tbptt_len", None))
    y_hat, y_true = simulate_seq(model, u_te, y_te, warmup=cfg.get("rec_warmup", 10))
    y_osa, _      = eval_seq(model, u_te, y_te)
    y_3s, y_true_3s = kstep_seq(model, u_te, y_te, k=3)
    res = dict(model="RNN", params=count_params(model),
               flops_per_sample=flops_per_sample(model),
               train_time_s=elapsed,
               nrmse_sim=nrmse(y_true, y_hat), vaf_sim=vaf(y_true, y_hat),
               nrmse_osa=nrmse(y_true, y_osa), vaf_osa=vaf(y_true, y_osa),
               nrmse_3step=nrmse(y_true_3s, y_3s), vaf_3step=vaf(y_true_3s, y_3s),
               history=history)
    print(f"test OSA NRMSE={res['nrmse_osa']:.4f}, VAF={res['vaf_osa']:.2f}% | "
          f"3-step NRMSE={res['nrmse_3step']:.4f}, VAF={res['vaf_3step']:.2f}% | "
          f"sim NRMSE={res['nrmse_sim']:.4f}, VAF={res['vaf_sim']:.2f}%, "
          f"train_time={elapsed:.1f}s, FLOPs/sample={res['flops_per_sample']:,}")
    plot_three_sims("RNN", y_true, y_hat, y_osa,
                    y_true_3s, y_3s,
                    res["vaf_sim"], res["vaf_osa"], res["vaf_3step"])
    plot_error_spectrum(y_true, y_hat, cfg["fs"], title="RNN -- error spectrum")
    return model, res


In [ ]:
def run_lstm(cfg):
    model = LSTMModel(hidden=cfg["rec_hidden"], layers=cfg["rec_layers"],
                       var_dropout=cfg.get("rec_var_dropout", 0.0))
    print(f"LSTM params: {count_params(model):,}")
    history, elapsed = train_seq(model, u_tr, y_tr, u_va, y_va,
                                 epochs=cfg["epochs_rec"],
                                 lr=cfg["lr"], weight_decay=cfg["weight_decay"],
                                 grad_clip=cfg.get("rec_grad_clip", 1.0),
                                 warmup=cfg.get("rec_warmup", 10),
                                 kstep_schedule=cfg.get("rec_kstep_schedule", ((0, 1),)),
                                 kstep_min_epochs=cfg.get("rec_kstep_min_epochs", None),
                                 tbptt_len=cfg.get("rec_tbptt_len", None))
    y_hat, y_true = simulate_seq(model, u_te, y_te, warmup=cfg.get("rec_warmup", 10))
    y_osa, _      = eval_seq(model, u_te, y_te)
    y_3s, y_true_3s = kstep_seq(model, u_te, y_te, k=3)
    res = dict(model="LSTM", params=count_params(model),
               flops_per_sample=flops_per_sample(model),
               train_time_s=elapsed,
               nrmse_sim=nrmse(y_true, y_hat), vaf_sim=vaf(y_true, y_hat),
               nrmse_osa=nrmse(y_true, y_osa), vaf_osa=vaf(y_true, y_osa),
               nrmse_3step=nrmse(y_true_3s, y_3s), vaf_3step=vaf(y_true_3s, y_3s),
               history=history)
    print(f"test OSA NRMSE={res['nrmse_osa']:.4f}, VAF={res['vaf_osa']:.2f}% | "
          f"3-step NRMSE={res['nrmse_3step']:.4f}, VAF={res['vaf_3step']:.2f}% | "
          f"sim NRMSE={res['nrmse_sim']:.4f}, VAF={res['vaf_sim']:.2f}%, "
          f"train_time={elapsed:.1f}s, FLOPs/sample={res['flops_per_sample']:,}")
    plot_three_sims("LSTM", y_true, y_hat, y_osa,
                    y_true_3s, y_3s,
                    res["vaf_sim"], res["vaf_osa"], res["vaf_3step"])
    plot_error_spectrum(y_true, y_hat, cfg["fs"], title="LSTM -- error spectrum")
    return model, res


In [ ]:
def run_gru(cfg):
    model = GRUModel(hidden=cfg["rec_hidden"], layers=cfg["rec_layers"],
                       var_dropout=cfg.get("rec_var_dropout", 0.0))
    print(f"GRU params: {count_params(model):,}")
    history, elapsed = train_seq(model, u_tr, y_tr, u_va, y_va,
                                 epochs=cfg["epochs_rec"],
                                 lr=cfg["lr"], weight_decay=cfg["weight_decay"],
                                 grad_clip=cfg.get("rec_grad_clip", 1.0),
                                 warmup=cfg.get("rec_warmup", 10),
                                 kstep_schedule=cfg.get("rec_kstep_schedule", ((0, 1),)),
                                 kstep_min_epochs=cfg.get("rec_kstep_min_epochs", None),
                                 tbptt_len=cfg.get("rec_tbptt_len", None))
    y_hat, y_true = simulate_seq(model, u_te, y_te, warmup=cfg.get("rec_warmup", 10))
    y_osa, _      = eval_seq(model, u_te, y_te)
    y_3s, y_true_3s = kstep_seq(model, u_te, y_te, k=3)
    res = dict(model="GRU", params=count_params(model),
               flops_per_sample=flops_per_sample(model),
               train_time_s=elapsed,
               nrmse_sim=nrmse(y_true, y_hat), vaf_sim=vaf(y_true, y_hat),
               nrmse_osa=nrmse(y_true, y_osa), vaf_osa=vaf(y_true, y_osa),
               nrmse_3step=nrmse(y_true_3s, y_3s), vaf_3step=vaf(y_true_3s, y_3s),
               history=history)
    print(f"test OSA NRMSE={res['nrmse_osa']:.4f}, VAF={res['vaf_osa']:.2f}% | "
          f"3-step NRMSE={res['nrmse_3step']:.4f}, VAF={res['vaf_3step']:.2f}% | "
          f"sim NRMSE={res['nrmse_sim']:.4f}, VAF={res['vaf_sim']:.2f}%, "
          f"train_time={elapsed:.1f}s, FLOPs/sample={res['flops_per_sample']:,}")
    plot_three_sims("GRU", y_true, y_hat, y_osa,
                    y_true_3s, y_3s,
                    res["vaf_sim"], res["vaf_osa"], res["vaf_3step"])
    plot_error_spectrum(y_true, y_hat, cfg["fs"], title="GRU -- error spectrum")
    return model, res


In [ ]:
# Save best-per-family model checkpoints.
from datetime import datetime as _dt_ckpt, timezone as _tz_ckpt

CKPT_DIR = Path("results/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PER_FAMILY = {"NNARX": -float("inf"), "RNN": -float("inf"),
                   "LSTM": -float("inf"), "GRU": -float("inf")}


def _val_vaf_for(family, model):
    if family == "NNARX":
        y_hat, y_true = eval_narx(model, u_va, y_va)
    else:
        y_hat, y_true = simulate_seq(model, u_va, y_va,
                                     warmup=CFG.get("rec_warmup", 10))
    return float(vaf(y_true, y_hat))


def _cfg_snapshot(family, model, cfg):
    if family == "NNARX":
        return {
            "kind":    "NNARX",
            "u_taps":  tuple(int(t) for t in model.u_taps),
            "y_taps":  tuple(int(t) for t in model.y_taps),
            "hidden":  tuple(int(h) for h in cfg["narx_hidden"]),
            "act":     cfg.get("narx_act", "tanh"),
            "dropout": float(cfg.get("narx_dropout", 0.0)),
        }
    return {
        "kind":        family,                 # "RNN" | "LSTM" | "GRU"
        "hidden":      int(cfg["rec_hidden"]),
        "layers":      int(cfg["rec_layers"]),
        "var_dropout": float(cfg.get("rec_var_dropout", 0.0)),
    }


def save_if_best(family, model, res, cfg):
    family = family.upper()
    assert family in BEST_PER_FAMILY, f"unknown family {family!r}"
    val_vaf = _val_vaf_for(family, model)
    if val_vaf > BEST_PER_FAMILY[family]:
        BEST_PER_FAMILY[family] = val_vaf
        metrics = {k: v for k, v in res.items() if k != "history"}
        metrics["val_vaf_sim"] = val_vaf
        bundle = {
            "family":        family,
            "state_dict":    model.state_dict(),
            "cfg":           _cfg_snapshot(family, model, cfg),
            "metrics":       metrics,
            "torch_version": torch.__version__,
            "saved_at":      _dt_ckpt.now(_tz_ckpt.utc).isoformat(),
        }
        path = CKPT_DIR / f"best_{family.lower()}.pt"
        torch.save(bundle, path)
        print(f"[CKPT] {family} <- val_vaf_sim={val_vaf:.2f}%  -> {path}")
    else:
        print(f"[CKPT] {family} skip val_vaf_sim={val_vaf:.2f}% "
              f"(best so far {BEST_PER_FAMILY[family]:.2f}%)")
    return val_vaf


### FROLS structure selection (sysidentpy)


In [ ]:
# Auto-install sysidentpy
try:
    from sysidentpy.model_structure_selection import FROLS
    from sysidentpy.basis_function import Polynomial
    from sysidentpy.parameter_estimation.estimators import LeastSquares
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sysidentpy"])
    from sysidentpy.model_structure_selection import FROLS
    from sysidentpy.basis_function import Polynomial
    from sysidentpy.parameter_estimation.estimators import LeastSquares


def _flat64(seqs):
    return np.concatenate([np.asarray(s, np.float64).reshape(-1, 1) for s in seqs])


def frols_select_lags(u_seqs, y_seqs, ylag_max=64, xlag_max=64, n_terms=20, degree=1):
    """Returns u_taps, y_taps, u_lags_s, y_lags_s, frols"""
    frols = FROLS(
        order_selection=True,
        n_info_values=n_terms,
        ylag=ylag_max,
        xlag=xlag_max,
        info_criteria="aic",
        estimator=LeastSquares(),
        basis_function=Polynomial(degree=degree),
    )
    frols.fit(X=_flat64(u_seqs), y=_flat64(y_seqs))
    u_lags_s, y_lags_s = [], []
    for term_codes in frols.final_model:
        for code in np.asarray(term_codes).flatten():
            code = int(code)
            if code == 0:
                continue
            var_id = code // 1000
            lag = code % 1000
            if var_id == 1:
                y_lags_s.append(lag)
            elif var_id == 2:
                u_lags_s.append(lag)
    u_lags_s = sorted(set(u_lags_s))
    y_lags_s = sorted(set(y_lags_s))
    u_taps = tuple(u_lags_s)
    y_taps = tuple(y_lags_s)
    print(f"FROLS: {len(u_taps)} u-lags + {len(y_taps)} y-lags selected")
    print(f"  u_taps={u_taps}")
    print(f"  y_taps={y_taps}")
    print(f"  ERR:    {frols.err.round(4)}")
    return u_taps, y_taps, u_lags_s, y_lags_s, frols


def run_nnarx_frols(cfg, ylag_max=64, xlag_max=64, n_frols_terms=20, tag="nnarx_frols"):
    u_taps, y_taps, _, _, _ = frols_select_lags(
        u_tr, y_tr,
        ylag_max=ylag_max, xlag_max=xlag_max, n_terms=n_frols_terms,
    )
    if not u_taps:
        u_taps = (1, 2, 4, 8)
    if not y_taps:
        y_taps = (1, 2, 4)
    cfg_f = dict(cfg)
    cfg_f["narx_u_taps"] = u_taps
    cfg_f["narx_y_taps"] = y_taps
    cfg_f["narx_nu"] = len(u_taps)
    cfg_f["narx_ny"] = len(y_taps)
    model, res = run_nnarx(cfg_f, tag=tag)
    res["model"] = "NNARX-FROLS"
    return model, res

### NNARX

In [ ]:
# NNARX main: FROLS K=20 (data-driven taps), promoted from ablation sweep
# after LOSO CV validation (NRMSE_sim 0.862 +/- 0.019 vs hand-crafted 0.91 +/- 0.04).
model_nnarx, RESULTS["nnarx"] = run_nnarx_frols(CFG, n_frols_terms=20, tag="nnarx")
RESULTS["nnarx"]["model"] = "NNARX (FROLS K=20)"
save_if_best("NNARX", model_nnarx, RESULTS["nnarx"], CFG)


### NNARX  sysidentpy-enhanced variants

In [ ]:
for _K in (10, 40):
    _m_frols, RESULTS[f"nnarx_frols_k{_K}"] = run_nnarx_frols(CFG, n_frols_terms=_K, tag=f"nnarx_frols_k{_K}")
    save_if_best("NNARX", _m_frols, RESULTS[f"nnarx_frols_k{_K}"], CFG)

### Simple RNN

In [ ]:
model_rnn, RESULTS["rnn"] = run_rnn(CFG)
save_if_best("RNN", model_rnn, RESULTS["rnn"], CFG)


### LSTM

In [ ]:
model_lstm, RESULTS["lstm"] = run_lstm(CFG)
save_if_best("LSTM", model_lstm, RESULTS["lstm"], CFG)


### GRU (optional in the brief)

In [ ]:
model_gru, RESULTS["gru"] = run_gru(CFG)
save_if_best("GRU", model_gru, RESULTS["gru"], CFG)


## 12. Phase 2 â€” Architecture sweep

In [ ]:
PHASE2 = []


def repeat_seeds(runner_fn, cfg, n_seeds=1, base_seed=SEED):
    if int(n_seeds) <= 1:
        m, r = runner_fn(cfg)
        return m, r
    all_res, last_model = [], None
    metric_keys = ("nrmse_sim", "vaf_sim", "nrmse_osa", "vaf_osa",
                   "nrmse_3step", "vaf_3step", "train_time_s")
    for s in range(int(n_seeds)):
        seed = int(base_seed) + s
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        cfg_s = dict(cfg)
        cfg_s["narx_seed_base"] = seed
        cfg_s["rec_seed_base"]  = seed
        print(f"\n[seed {s+1}/{n_seeds}  base+offset={seed}]")
        m, r = runner_fn(cfg_s)
        all_res.append(r); last_model = m
    agg = dict(all_res[0])
    agg.pop("history", None)
    for k in metric_keys:
        vals = [float(r[k]) for r in all_res
                if (k in r) and (r[k] is not None) and not math.isnan(float(r[k]))]
        if not vals:
            continue
        agg[k]            = float(np.mean(vals))
        agg[k + "_std"]   = float(np.std(vals, ddof=0))
    agg["n_seeds"] = int(n_seeds)
    return last_model, agg

In [ ]:
# Phase 2 -- RNN sweep over (hidden, layers)
for hidden in (32, 64, 128):
    for layers in (1, 2):
        cfg = dict(CFG); cfg["rec_hidden"] = hidden; cfg["rec_layers"] = layers
        cfg["epochs_rec"] = 100
        print(f"\n--- RNN hidden={hidden} layers={layers} ---")
        n_seeds = int(cfg.get("phase2_seeds", 1))
        last_model, res = repeat_seeds(run_rnn, cfg, n_seeds=n_seeds)
        res["kind"] = "rnn"; res["hidden"] = hidden; res["layers"] = layers
        save_if_best("RNN", last_model, res, cfg)
        PHASE2.append(res)

In [ ]:
# Phase 2 -- LSTM sweep over (hidden, layers)
for hidden in (32, 64, 128):
    for layers in (1, 2):
        cfg = dict(CFG); cfg["rec_hidden"] = hidden; cfg["rec_layers"] = layers
        cfg["epochs_rec"] = 500
        print(f"\n--- LSTM hidden={hidden} layers={layers} ---")
        n_seeds = int(cfg.get("phase2_seeds", 1))
        last_model, res = repeat_seeds(run_lstm, cfg, n_seeds=n_seeds)
        res["kind"] = "lstm"; res["hidden"] = hidden; res["layers"] = layers
        save_if_best("LSTM", last_model, res, cfg)
        PHASE2.append(res)

In [ ]:
 # Phase 2 -- GRU sweep over (hidden, layers)
for hidden in (32, 64, 128):
    for layers in (1, 2):
        cfg = dict(CFG); cfg["rec_hidden"] = hidden; cfg["rec_layers"] = layers
        cfg["epochs_rec"] = 100
        print(f"\n--- GRU hidden={hidden} layers={layers} ---")
        n_seeds = int(cfg.get("phase2_seeds", 1))
        last_model, res = repeat_seeds(run_gru, cfg, n_seeds=n_seeds)
        res["kind"] = "gru"; res["hidden"] = hidden; res["layers"] = layers
        save_if_best("GRU", last_model, res, cfg)
        PHASE2.append(res)

In [ ]:
print("\nPhase 2 recurrent summary (sorted by sim NRMSE):")
for r in sorted(PHASE2, key=lambda r: r["nrmse_sim"]):
    sd = r.get("nrmse_sim_std")
    vsd = r.get("vaf_sim_std")
    sd_s = f" Â±{sd:.3f}" if sd is not None else ""
    vsd_s = f" Â±{vsd:.1f}" if vsd is not None else ""
    ns = r.get("n_seeds", 1)
    print(f"{r['model']:<6} h={r['hidden']:3d} L={r['layers']}  "
          f"NRMSE={r['nrmse_sim']:.3f}{sd_s}  VAF={r['vaf_sim']:.1f}%{vsd_s}  "
          f"params={r['params']:,}  flops={r['flops_per_sample']:,}  "
          f"t={r['train_time_s']:.1f}s  seeds={ns}")

Path("results").mkdir(exist_ok=True)
with open("results/phase2_rec.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k != "history"} for r in PHASE2], f, indent=2)
print("saved: results/phase2_rec.json")

### Phase 2 â€” NNARX hidden-capacity sweep (FROLS K=20 taps)

In [ ]:
PHASE2_NARX = []

_p2_u_taps, _p2_y_taps, _, _, _ = frols_select_lags(
    u_tr, y_tr, ylag_max=64, xlag_max=64, n_terms=20, degree=1)
print(f"Phase 2 FROLS taps: u_taps={_p2_u_taps}  y_taps={_p2_y_taps}")

_narx_hidden_configs = [(32, 32), (64, 64), (128, 128)]

for hid in _narx_hidden_configs:
    cfg = dict(CFG)
    cfg["narx_u_taps"] = _p2_u_taps; cfg["narx_y_taps"] = _p2_y_taps
    cfg["narx_nu"]     = len(_p2_u_taps); cfg["narx_ny"] = len(_p2_y_taps)
    cfg["narx_hidden"] = hid
    cfg["epochs_narx"]         = 100
    cfg["narx_ensemble"]       = 1
    cfg["narx_kstep_schedule"] = ((0, 1), (15, 5), (35, 15), (60, 40))
    run_label = f"frols-k20 h={list(hid)}"
    print(f"\n--- NNARX {run_label} ---")
    n_seeds = int(cfg.get("phase2_seeds", 1))
    last_model, res = repeat_seeds(run_nnarx, cfg, n_seeds=n_seeds)
    res["label"] = "frols-k20"; res["hidden_cfg"] = list(hid)
    save_if_best("NNARX", last_model, res, cfg)
    PHASE2_NARX.append(res)

print("\nPhase 2 NNARX summary (sorted by sim NRMSE):")
for r in sorted(PHASE2_NARX, key=lambda r: r["nrmse_sim"]):
    sd = r.get("nrmse_sim_std")
    vsd = r.get("vaf_sim_std")
    sd_s = f" Â±{sd:.3f}" if sd is not None else ""
    vsd_s = f" Â±{vsd:.1f}" if vsd is not None else ""
    ns = r.get("n_seeds", 1)
    print(f"NNARX {r['label']:<10} h={r['hidden_cfg']}  "
          f"NRMSE={r['nrmse_sim']:.3f}{sd_s}  VAF={r['vaf_sim']:.1f}%{vsd_s}  "
          f"params={r['params']:,}  flops={r['flops_per_sample']:,}  "
          f"t={r['train_time_s']:.1f}s  seeds={ns}")


## 13. Phase 3 â€” Comparison across model families

Head-to-head table and a simple params-vs-accuracy scatter.

In [ ]:
rows = []

for key in ("nnarx", "nnarx_frols_k10", "nnarx_frols_k40", "rnn", "lstm", "gru"):
    if key not in RESULTS: continue
    r = RESULTS[key]
    name = r["model"]
    if r.get("free_run_diverged") or r.get("three_step_diverged"):
        name = name + "*"
    rows.append([name, r["params"], r["flops_per_sample"], r["train_time_s"],
                 r.get("nrmse_osa",   float("nan")), r.get("vaf_osa",   float("nan")),
                 r.get("nrmse_3step", float("nan")), r.get("vaf_3step", float("nan")),
                 r["nrmse_sim"],     r["vaf_sim"]])

hdr = (f"{'model':<14} {'params':>10} {'flops/smp':>12} {'t_train[s]':>10} "
       f"{'NRMSE_osa':>10} {'VAF_osa[%]':>11} "
       f"{'NRMSE_3st':>10} {'VAF_3st[%]':>11} "
       f"{'NRMSE_sim':>10} {'VAF_sim[%]':>11}")
print(hdr)
for name, p, fl, t, n_o, v_o, n_3, v_3, n_s, v_s in rows:
    print(f"{name:<14} {p:>10,} {fl:>12,} {t:>10.1f} "
          f"{n_o:>10.3f} {v_o:>11.2f} {n_3:>10.3f} {v_3:>11.2f} "
          f"{n_s:>10.3f} {v_s:>11.2f}")
if any(name.endswith("*") for name, *_ in rows):
    print("* = guard-clipped free-run / 3-step (see *_diverged flag in JSON)")

Path("results").mkdir(exist_ok=True)
with open("results/phase1_metrics.json", "w") as out:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != "history"}
               for k, v in RESULTS.items()}, out, indent=2)
print("\nsaved: results/phase1_metrics.json")

fig, ax = plt.subplots(figsize=(8, 4))
for row in rows:
    name, params, _flops, _t = row[0], row[1], row[2], row[3]
    n_sim = row[-2]
    ax.scatter(params, n_sim, s=80)
    ax.annotate(name, (params, n_sim), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("parameters"); ax.set_ylabel("sim NRMSE")
ax.set_title("Complexity vs simulation accuracy (test set)")
plt.tight_layout(); plt.show()

In [ ]:

RUN_LOSO_CV = True

def _best_arch_overrides(family):
    """Return cfg-key overrides for the best architecture found in Phase 2."""
    path = CKPT_DIR / f"best_{family.lower()}.pt"
    if not path.exists():
        return {}
    bundle = torch.load(path, map_location="cpu", weights_only=False)
    snap = bundle["cfg"]
    if family == "NNARX":
        return {"narx_hidden": tuple(snap["hidden"])}
    return {"rec_hidden": snap["hidden"], "rec_layers": snap["layers"]}

if RUN_LOSO_CV:
    assert CFG["split"] == "loso" and CFG["dataset"] == "medium",         "Outer LOSO loop expects dataset='medium', split='loso'."

    # Log which architectures will be used
    for _fam in ("NNARX", "RNN", "LSTM", "GRU"):
        _ov = _best_arch_overrides(_fam)
        if _ov:
            print(f"[LOSO] {_fam} best arch: {_ov}")
        else:
            print(f"[LOSO] {_fam} no checkpoint found, using CFG defaults")

    fold_rows = []
    for s in range(S):
        cfg = dict(CFG); cfg["loso_test_subject"] = s
        (u_tr, y_tr), (u_va, y_va), (u_te, y_te) = make_loso_split(u_all, y_all, cfg)
        print(f"===== LOSO fold: test subject = {s} | "
              f"train {u_tr.shape}, val {u_va.shape}, test {u_te.shape} =====")

        cfg_narx = dict(cfg); cfg_narx.update(_best_arch_overrides("NNARX"))
        cfg_rnn  = dict(cfg); cfg_rnn.update(_best_arch_overrides("RNN"))
        cfg_lstm = dict(cfg); cfg_lstm.update(_best_arch_overrides("LSTM"))
        cfg_gru  = dict(cfg); cfg_gru.update(_best_arch_overrides("GRU"))

        _, r_narx = run_nnarx_frols(cfg_narx, n_frols_terms=20, tag=f"nnarx_s{s}")
        r_narx["model"] = "NNARX"
        _, r_rnn  = run_rnn(cfg_rnn)
        _, r_lstm = run_lstm(cfg_lstm)
        _, r_gru  = run_gru(cfg_gru)
        for r in (r_narx, r_rnn, r_lstm, r_gru):
            fold_rows.append({"fold": s, **{k: v for k, v in r.items() if k != "history"}})

    import statistics as stats
    print("LOSO summary (mean ± pstdev across folds):")
    by_model = {}
    for row in fold_rows:
        by_model.setdefault(row["model"], []).append(row)
    def _fmt(xs, pct=False):
        if not xs:
            return "    n/a       "
        m = stats.mean(xs); s = stats.pstdev(xs)
        return f"{m:6.2f} ± {s:5.2f}{'%' if pct else ' '}"
    any_diverged = False
    for name, rows in by_model.items():
        nr_s = [r["nrmse_sim"]   for r in rows]
        vf_s = [r["vaf_sim"]     for r in rows]
        nr_o = [r["nrmse_osa"]   for r in rows if "nrmse_osa"   in r]
        vf_o = [r["vaf_osa"]     for r in rows if "vaf_osa"     in r]
        nr_3 = [r["nrmse_3step"] for r in rows if "nrmse_3step" in r]
        vf_3 = [r["vaf_3step"]   for r in rows if "vaf_3step"   in r]
        flagged = any(r.get("free_run_diverged") or r.get("three_step_diverged")
                      for r in rows)
        any_diverged = any_diverged or flagged
        tag = name + ("*" if flagged else "")
        print(f"  {tag:<8} "
              f"OSA  NRMSE {_fmt(nr_o)} VAF {_fmt(vf_o, pct=True)} | "
              f"3-step NRMSE {_fmt(nr_3)} VAF {_fmt(vf_3, pct=True)} | "
              f"sim NRMSE {_fmt(nr_s)} VAF {_fmt(vf_s, pct=True)}")
    if any_diverged:
        print("* = at least one fold guard-clipped (see *_diverged flag in JSON)")

    # ---------------- Compact aggregate table (matches Phase-3 layout) ----------------
    def _cell(xs, pct=False, w=13):
        if not xs:
            return f"{'n/a':>{w}}"
        m = stats.mean(xs); sd = stats.pstdev(xs)
        txt = f"{m:.2f}±{sd:.2f}{'%' if pct else ''}"
        return f"{txt:>{w}}"

    print("LOSO compact table (mean ± pstdev across folds):")
    hdr = (f"{'model':<10} {'params':>10} {'flops/smp':>12} {'t_train[s]':>12} "
           f"{'NRMSE_osa':>13} {'VAF_osa':>13} "
           f"{'NRMSE_3st':>13} {'VAF_3st':>13} "
           f"{'NRMSE_sim':>13} {'VAF_sim':>13}")
    print(hdr)
    for name, rows in by_model.items():
        flagged = any(r.get("free_run_diverged") or r.get("three_step_diverged")
                      for r in rows)
        tag = name + ("*" if flagged else "")
        mp = stats.mean([r["params"]           for r in rows])
        mf = stats.mean([r["flops_per_sample"] for r in rows])
        mt = stats.mean([r["train_time_s"]     for r in rows])
        nr_s = [r["nrmse_sim"]   for r in rows]
        vf_s = [r["vaf_sim"]     for r in rows]
        nr_o = [r["nrmse_osa"]   for r in rows if "nrmse_osa"   in r]
        vf_o = [r["vaf_osa"]     for r in rows if "vaf_osa"     in r]
        nr_3 = [r["nrmse_3step"] for r in rows if "nrmse_3step" in r]
        vf_3 = [r["vaf_3step"]   for r in rows if "vaf_3step"   in r]
        print(f"{tag:<10} {int(round(mp)):>10,} {int(round(mf)):>12,} {mt:>12.1f} "
              f"{_cell(nr_o)} {_cell(vf_o, pct=True)} "
              f"{_cell(nr_3)} {_cell(vf_3, pct=True)} "
              f"{_cell(nr_s)} {_cell(vf_s, pct=True)}")

    Path("results").mkdir(exist_ok=True)
    with open("results/loso_folds.json", "w") as f:
        json.dump(fold_rows, f, indent=2)
    print("saved: results/loso_folds.json")
else:
    print("RUN_LOSO_CV=False — single-fold only. Set True to run all S folds.")


In [ ]:
# Verify per-family checkpoints round-trip: reload, recompute val VAF, compare.
def _reload_ckpt(path):
    bundle = torch.load(path, map_location=DEVICE, weights_only=False)
    snap = bundle["cfg"]
    kind = snap["kind"]
    if kind == "NNARX":
        m = NNARX(snap["u_taps"], snap["y_taps"],
                  hidden=tuple(snap["hidden"]),
                  act=snap["act"], dropout=snap["dropout"])
    elif kind == "RNN":
        m = RNNModel(hidden=snap["hidden"], layers=snap["layers"])
    elif kind == "LSTM":
        m = LSTMModel(hidden=snap["hidden"], layers=snap["layers"],
                      var_dropout=snap["var_dropout"])
    elif kind == "GRU":
        m = GRUModel(hidden=snap["hidden"], layers=snap["layers"],
                     var_dropout=snap["var_dropout"])
    else:
        raise ValueError(f"unknown kind {kind!r}")
    m.load_state_dict(bundle["state_dict"])
    m.to(DEVICE).eval()
    return m, bundle


_ckpts = sorted(CKPT_DIR.glob("best_*.pt"))
print(f"Found {len(_ckpts)} checkpoints in {CKPT_DIR}/:")
for _p in _ckpts:
    _family = _p.stem.replace("best_", "").upper()
    _model, _bundle = _reload_ckpt(_p)
    _saved = float(_bundle["metrics"]["val_vaf_sim"])
    _recomp = _val_vaf_for(_family, _model)
    _ok = abs(_recomp - _saved) < 1e-3
    print(f"  {_p.name:<20} saved val_vaf_sim={_saved:7.3f}%  "
          f"recomputed={_recomp:7.3f}%  "
          f"[{'OK' if _ok else 'MISMATCH'}]")
    if _family in BEST_PER_FAMILY:
        assert abs(_saved - BEST_PER_FAMILY[_family]) < 1e-9, (
            f"{_family} BEST_PER_FAMILY {BEST_PER_FAMILY[_family]} != bundle {_saved}")
